In [31]:
import os
import sys
from pathlib import Path
import cvxpy as cp
import joblib

ROOT_DIR = Path.cwd()
if (ROOT_DIR / 'experiments').is_dir() and (ROOT_DIR / 'scheduling').is_dir():
    pass
elif ROOT_DIR.name == 'experiments' and (ROOT_DIR.parent / 'scheduling').is_dir():
    ROOT_DIR = ROOT_DIR.parent

if ROOT_DIR != Path.cwd():
    os.chdir(ROOT_DIR)

sys.path.insert(0, str(ROOT_DIR))

import andes
import numpy as np
import csv

from experiments.run_sim_extract_ed import _load_yaml, _repeat_or_validate, add_measurement_devices
from scheduling.mtlsh_convex import compute_feature_bounds_from_training_data
from scheduling.mtlsh_convex import build_mtlsh_convex_constraints
from scheduling.economic_dispatch import ed_calculation
from data_generation.extract_metrics import build_feature_row, extract_simulation_row, export_plotter_all
from models.models import MTLSharedHeads
import torch


In [32]:
config_path = ROOT_DIR / 'experiments' / 'generation.yaml'
cost_config_path = ROOT_DIR / 'scheduling' / 'mtlsh_convex.yaml'
base_scale = 1.0
step_scale = 0.9

cfg = _load_yaml(Path(config_path))
cost_cfg = _load_yaml(Path(cost_config_path))
if 'ed_costs' not in cost_cfg:
    raise KeyError('Missing ed_costs in cost-config YAML.')


In [33]:
case_path = cfg['case']
ss = andes.load(case_path, setup=False)
ss.config.freq = float(50)
add_measurement_devices(ss)

rng = np.random.default_rng(int(cfg.get('seed', 42)))
regcv1_ids = ss.REGCV1.name.v
M_vec = rng.uniform(cfg['ibr']['M_range'][0], cfg['ibr']['M_range'][1], size=len(regcv1_ids))
D_vec = rng.uniform(cfg['ibr']['D_range'][0], cfg['ibr']['D_range'][1], size=len(regcv1_ids))

for uid in range(ss.PQ.n):
    ss.PQ.p0.v[uid] = ss.PQ.p0.v[uid] * base_scale
    ss.PQ.q0.v[uid] = ss.PQ.q0.v[uid] * base_scale
for uid in range(ss.PV.n):
    ss.PV.p0.v[uid] = ss.PV.p0.v[uid] * base_scale
    ss.PV.q0.v[uid] = ss.PV.q0.v[uid] * base_scale

ss.REGCV1.M.v, ss.REGCV1.D.v = M_vec, D_vec

ss.PQ.config.p2p = 1
ss.PQ.config.q2q = 1
ss.PQ.config.p2z = 0
ss.PQ.config.q2z = 0
ss.PQ.config.p2i = 0
ss.PQ.config.q2i = 0
ss.PQ.config.pq2z = 0

pq_p_before = np.asarray(ss.PQ.p0.v, dtype=float).copy()
pq_q_before = np.asarray(ss.PQ.q0.v, dtype=float).copy()
pq_names = list(ss.PQ.name.v) if ss.PQ.n else []

M_agg = np.mean(np.concatenate([ss.GENROU.M.v, ss.REGCV1.M.v])).sum()
D_agg = np.mean(np.concatenate([ss.GENROU.D.v, ss.REGCV1.D.v])).sum()

features = build_feature_row(
    base_load_scale=base_scale,
    load_step_scale=step_scale,
    load_step_time=float(cfg['tds']['load_step_time']),
    pq_names=pq_names,
    pq_p_before=pq_p_before,
    pq_q_before=pq_q_before,
    pq_p_after=pq_p_before * step_scale,
    pq_q_after=pq_q_before * step_scale,
    M_vec=M_vec,
    D_vec=D_vec,
    M_agg=M_agg,
    D_agg=D_agg,
)

x_features = cost_cfg.get('features', {}).get('x_features') or list(features.keys())


In [34]:
cost_cfg.setdefault('features', {})['x_features'] = x_features
try:
    x_min, x_max, _ = compute_feature_bounds_from_training_data(cost_cfg)
    print(f'Computed bounds for {len(x_features)} features')
except Exception as exc:
    x_min, x_max = None, None
    print(f'Bounds computation skipped/failed: {exc}')

feat_vec = np.array([features[name] for name in x_features], dtype=float).reshape(1, -1)
x_scaler_path = cost_cfg.get('scalers', {}).get('x_scaler_path')
if x_scaler_path:
    x_scaler = joblib.load(x_scaler_path)
    feat_scaled = x_scaler.transform(feat_vec)
else:
    feat_scaled = feat_vec

x_link = cp.Constant(feat_scaled.reshape(-1))

x_nn, y_nn, nn_constraints = build_mtlsh_convex_constraints(cost_cfg)
nn_constraints.append(x_nn == x_link)
print(f'x_link shape: {x_link.shape}')


Computed bounds for 53 features
x_link shape: (53,)


/Applications/anaconda3/envs/GESTURE/lib/python3.9/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [35]:
# model_cfg = cost_cfg['model']
# state_path = Path(model_cfg['state_dict'])
# if state_path.is_dir():
#     state_path = state_path / 'vis_mlp_state_dict.pt'

# model = MTLSharedHeads(
#     in_dim=int(model_cfg['in_dim']),
#     n_tasks=int(model_cfg['n_tasks']),
#     shared_sizes=model_cfg['shared_sizes'],
#     head_sizes=model_cfg['head_sizes'],
#     dropout=float(model_cfg.get('dropout', 0.0)),
# )
# state = torch.load(state_path, map_location='cpu')
# model.load_state_dict(state)
# model.eval()

# with torch.no_grad():
#     pred_norm = model(torch.from_numpy(feat_scaled.astype(np.float32)).to("cpu")).cpu().numpy()

# y_scaler_path = cost_cfg.get('scalers', {}).get('y_scaler_path')
# if y_scaler_path:
#     y_scaler = joblib.load(y_scaler_path)
#     pred = y_scaler.inverse_transform(pred_norm)
# else:
#     pred = pred_norm

# y_features = cost_cfg.get('features', {}).get('y_features')
# if y_features and len(y_features) == pred.shape[1]:
#     pred_dict = {name: float(val) for name, val in zip(y_features, pred.flatten())}
#     pred_norm_dict = {name: float(val) for name, val in zip(y_features, pred_norm.flatten())}
#     print('pred (unscaled):', pred_dict)
#     print('pred (scaled):', pred_norm_dict)
# else:
#     print('pred (unscaled):', pred.flatten())
#     print('pred (scaled):', pred_norm.flatten())

In [36]:
# # Check scaled preds vs scaled bounds
# y_min = np.array(cost_cfg["bounds"].get("y_min", []), dtype=float)
# y_max = np.array(cost_cfg["bounds"].get("y_max", []), dtype=float)

# print("pred_norm:", pred_norm.flatten())
# print("y_min:", y_min)
# print("y_max:", y_max)
# print("violations:",
#       np.any(pred_norm.flatten() < y_min) or np.any(pred_norm.flatten() > y_max))

In [37]:
import andes
import numpy as np
import cvxpy as cp
from typing import Optional, Sequence



def economic_dispatch_quad(
    Pd: float,
    a: np.ndarray,
    b: np.ndarray,
    c: np.ndarray,
    Pg_min: np.ndarray,
    Pg_max: np.ndarray,
    *,
    extra_constraints: Optional[Sequence[cp.Constraint]] = None,
    solver: Optional[str] = "GUROBI",
    solver_kwargs: Optional[dict] = None,
):
    """
    Solve a standard quadratic economic dispatch problem:

        minimize   sum_i ( a_i + b_i * Pg_i + c_i * Pg_i^2 )
        s.t.       sum_i Pg_i = Pd
                   Pg_min_i <= Pg_i <= Pg_max_i

    Parameters
    ----------
    Pd : float
        Total active power demand (pu or MW, consistent with coefficients).
    a, b, c : np.ndarray
        Cost coefficients for each generator (same length).
    Pg_min, Pg_max : np.ndarray
        Min and max active power outputs for each generator.

    extra_constraints : Sequence[cp.Constraint], optional
        Additional convex (or MIP) constraints that may include extra variables.
        Use this to couple Pg to other models (e.g., NN constraints).
    solver : str, optional
        CVXPY solver name. Defaults to OSQP.
    solver_kwargs : dict, optional
        Extra keyword arguments passed to cvxpy.Problem.solve().

    Returns
    -------
    Pg_opt : np.ndarray
        Optimal generator outputs.
    total_cost : float
        Optimal total generation cost.
    lam : float or None
        Lagrange multiplier (system marginal cost) on the power balance.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    c = np.asarray(c, dtype=float)
    Pg_min = np.asarray(Pg_min, dtype=float)
    Pg_max = np.asarray(Pg_max, dtype=float)

    ng = len(a)
    assert all(len(x) == ng for x in (b, c, Pg_min, Pg_max)), \
        "Cost and limit arrays must have the same length."

    # Decision variable
    Pg = cp.Variable(ng, name="Pg")

    # Objective
    cost_expr = a + cp.multiply(b, Pg) + cp.multiply(c, cp.square(Pg))
    objective = cp.Minimize(cp.sum(cost_expr))

    # Constraints
    constraints = [
        cp.sum(Pg) == Pd,
        Pg >= Pg_min,
        Pg <= Pg_max,
    ]
    if extra_constraints:
        constraints += list(extra_constraints)

    global prob 
    
    prob = cp.Problem(objective, constraints)
    solve_kwargs = solver_kwargs or {}
    if solver is None:
        prob.solve(**solve_kwargs)
    else:
        prob.solve(solver=solver, **solve_kwargs)

    if prob.status not in ("optimal", "optimal_inaccurate"):
        raise RuntimeError(f"Economic dispatch failed with status: {prob.status}")

    Pg_opt = Pg.value
    total_cost = prob.value

    lam = None
    try:
        lam = -float(constraints[0].dual_value)
    except Exception:
        pass

    return Pg_opt, total_cost, lam


def ed_calculation(
    ss: andes.System,
    a: np.ndarray,
    b: np.ndarray,
    c: np.ndarray,
    *,
    extra_constraints: Optional[Sequence[cp.Constraint]] = None,
    solver: Optional[str] = "GUROBI",
    solver_kwargs: Optional[dict] = None,
):
    """
    Solve a *lossless* quadratic ED using total PQ demand and apply
    the optimal Pg setpoints to Slack/PV generators *before* ss.setup().

    This allows dynamic models (TGOV1, GENROU, etc.) to be initialized
    consistently with the ED dispatch.

    Parameters
    ----------
    ss : andes.System (loaded with setup=False)
    a, b, c : np.ndarray
        Cost coefficients, aligned with ss.GENROU rows.
    Pg_min, Pg_max : np.ndarray
        Min/max Pg for each GENROU, same length as a,b,c.

    extra_constraints : Sequence[cp.Constraint], optional
        Additional convex (or MIP) constraints that may include extra variables.
        Use this to couple Pg to other models (e.g., NN constraints).
    solver : str, optional
        CVXPY solver name. Defaults to OSQP.
    solver_kwargs : dict, optional
        Extra keyword arguments passed to cvxpy.Problem.solve().

    Returns
    -------
    Pg_opt : np.ndarray
    total_cost : float
    lam : float
    """
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    c = np.asarray(c, float)
    Pg_min, Pg_max = get_pg_limits_from_static(ss)

    ng = ss.PV.n + ss.Slack.n
    assert ng == len(a) == len(b) == len(c) == len(Pg_min) == len(Pg_max), \
        "Cost and limit arrays must match number of GENROU units."

    # Total active demand (lossless ED)
    Pd = float(np.sum(ss.PQ.p0.v))

    # Solve ED for ALL generators (including the one at the slack bus)
    Pg_opt, total_cost, lam = economic_dispatch_quad(
        Pd,
        a,
        b,
        c,
        Pg_min,
        Pg_max,
        extra_constraints=extra_constraints,
        solver=solver,
        solver_kwargs=solver_kwargs,
    )

    # Map GENROU units to static generators (Slack / PV) by bus
    gen_buses = np.asarray(ss.PV.bus.v, dtype=int)
    slack_map = {bus: i for i, bus in enumerate(ss.Slack.bus.v)} if ss.Slack.n > 0 else {}
    pv_map    = {bus: i for i, bus in enumerate(ss.PV.bus.v)}    if ss.PV.n > 0    else {}

    for Pg_star, bus in zip(Pg_opt, gen_buses):
        if bus in slack_map:
            j = slack_map[bus]
            ss.Slack.p0.v[j] = Pg_star
        elif bus in pv_map:
            j = pv_map[bus]
            ss.PV.p0.v[j] = Pg_star
        else:
            raise RuntimeError(
                f"No Slack or PV generator at bus {bus} for GENROU; cannot apply ED."
            )

    return Pg_opt, total_cost, lam


def get_pg_limits_from_static(ss):
    """
    Build Pg_min and Pg_max arrays aligned with ss.GENROU order,
    using PV.pmin/pmax and Slack.pmin/pmax.
    """
    Pg_min = [ss.PV.pmin.v[i] for i in range(ss.PV.n)]
    Pg_max = [ss.PV.pmax.v[i] for i in range(ss.PV.n)]
    
    Pg_min.extend([ss.Slack.pmin.v[i] for i in range(ss.Slack.n)])
    Pg_max.extend([ss.Slack.pmax.v[i] for i in range(ss.Slack.n)])

    return Pg_min, Pg_max

In [38]:
a = _repeat_or_validate(cost_cfg['ed_costs']['a'], ss.PV.n + ss.Slack.n, 'ed_costs.a')
b = _repeat_or_validate(cost_cfg['ed_costs']['b'], ss.PV.n + ss.Slack.n, 'ed_costs.b')
c = _repeat_or_validate(cost_cfg['ed_costs']['c'], ss.PV.n + ss.Slack.n, 'ed_costs.c')

print("nn_constraints count:", len(nn_constraints))
print("sample constraints:", nn_constraints[:3])

prob = None 

Pg_opt, ed_cost, ed_lam = ed_calculation(
    ss,
    a=a,
    b=b,
    c=c,
    extra_constraints=nn_constraints,
    solver='Gurobi',
)

print('ED cost:', ed_cost)
print('ED lambda:', ed_lam)
print('Pg_opt:', Pg_opt)

nn_constraints count: 47
sample constraints: [Inequality(Constant(CONSTANT, ZERO, ())), Inequality(Expression(AFFINE, UNKNOWN, (256,))), Inequality(Constant(CONSTANT, ZERO, ()))]
ED cost: 1834.4483600002059
ED lambda: 99.68000000085064
Pg_opt: [10.4    4.484  4.484  4.484  4.484  6.97   4.484  5.64   8.65   4.484]


In [39]:
name_to_var = {v.name(): v for v in prob.variables()}
print(name_to_var.get("var3369"))
for i, c in enumerate(nn_constraints):
    if any(v.name() == "var3357" for v in c.variables()):
        print(i, c)

None


In [40]:
for i, c in enumerate(nn_constraints):
    print(f"[{i}] {c}")
for i, c in enumerate(nn_constraints):
    print(f"[{i}] {repr(c)}")

vars_ = prob.variables()
print(len(vars_))
for v in vars_:
    print(v, v.shape, id(v))

[0] 0.0 <= layer0 (shared)
[1] [[0.20 0.11 ... -0.04 -0.09]
 [0.19 0.03 ... -0.00 -0.09]
 ...
 [-0.04 0.06 ... -0.02 0.00]
 [-0.03 0.05 ... 0.09 -0.07]] @ features + [ 5.93613759e-02 -4.72251400e-02  1.32986799e-01 -1.29844740e-01
 -8.96802843e-02  2.45431855e-01 -1.54128551e-01  2.13530492e-02
  1.32052511e-01  3.70800570e-02  2.50539035e-01 -1.52488708e-01
  7.03447324e-04 -1.29856542e-01  1.52599469e-01  2.58521080e-01
 -1.93848342e-01 -1.22942597e-01 -1.52826220e-01 -9.82594118e-02
  6.77709877e-02  2.80491896e-02 -1.69440389e-01 -5.43986559e-02
 -1.45246059e-01  3.67481224e-02 -1.32801443e-01 -1.89121589e-01
 -2.12125927e-01  1.08150005e-01  1.62238061e-01 -1.66167114e-02
  1.18188187e-02 -4.48790751e-02 -4.07839119e-02  8.53254721e-02
 -1.17577016e-01 -2.40044873e-02  2.32777432e-01 -1.76383704e-01
  9.99534503e-02  1.90189555e-01  1.05085410e-01 -2.61394344e-02
 -2.68016569e-02 -1.05583601e-01 -7.35452846e-02 -2.23238051e-01
 -4.69135530e-02  6.56568930e-02 -1.53620243e-01  1.19

In [41]:
a = _repeat_or_validate(cost_cfg['ed_costs']['a'], ss.PV.n + ss.Slack.n, 'ed_costs.a')
b = _repeat_or_validate(cost_cfg['ed_costs']['b'], ss.PV.n + ss.Slack.n, 'ed_costs.b')
c = _repeat_or_validate(cost_cfg['ed_costs']['c'], ss.PV.n + ss.Slack.n, 'ed_costs.c')

Pg_opt, ed_cost, ed_lam = ed_calculation(
    ss,
    a=a,
    b=b,
    c=c,
    # extra_constraints=nn_constraints,
    solver='GUROBI',
)

print('ED cost:', ed_cost)
print('ED lambda:', ed_lam)
print('Pg_opt:', Pg_opt)

ED cost: 1834.4483600002059
ED lambda: 99.68000000085064
Pg_opt: [10.4    4.484  4.484  4.484  4.484  6.97   4.484  5.64   8.65   4.484]


In [42]:
ss.setup()
ss.PFlow.run()

ss.TDS.config.no_tqdm = bool(cfg['tds'].get('no_tqdm', True))
ss.TDS.config.criteria = int(cfg['tds'].get('criteria', 0))
ss.TDS.config.tol = float(cfg['tds'].get('tol', 1e-6))
ss.TDS.config.tf = float(cfg['tds']['t_end'])
ss.TDS.config.tstep = float(cfg['tds']['t_step'])
ss.TDS.config.fixt = int(cfg['tds'].get('fixt', 0))
ss.TDS.config.method = str(cfg['tds'].get('method', 'backeuler'))
ss.TDS.config.honest = int(cfg['tds'].get('honest', 0))
ss.TDS.config.max_iter = int(cfg['tds'].get('max_iter', 35))
ss.TDS.config.shrinkt = int(cfg['tds'].get('shrinkt', 1))

ss.TDS.init()

for uid in range(ss.PQ.n):
    p = ss.PQ.p0.v[uid] * step_scale
    q = ss.PQ.q0.v[uid] * step_scale
    ss.PQ.p0.v[uid], ss.PQ.Ppf.v[uid] = p, p
    ss.PQ.q0.v[uid], ss.PQ.Qpf.v[uid] = q, q

success = bool(ss.TDS.run())
ss.TDS.load_plotter()

pq_p_after = np.asarray(ss.PQ.Ppf.v, dtype=float).copy()
pq_q_after = np.asarray(ss.PQ.Qpf.v, dtype=float).copy()

plotter_csv = None
plotter_cfg = cfg.get('plotter', {})
export_plotter = bool(plotter_cfg.get('export', False))
plotter_dir = Path(cfg.get('output_dir', 'experiments')) / plotter_cfg.get('subdir', 'plotter') if export_plotter else None
if export_plotter and plotter_dir is not None:
    plotter_dir.mkdir(parents=True, exist_ok=True)
    plotter_csv = str(plotter_dir / 'plotter_single.csv')
    export_plotter_all(ss.TDS.plotter, plotter_csv)

row = extract_simulation_row(
    ss=ss,
    base_load_scale=base_scale,
    load_step_scale=step_scale,
    load_step_time=float(cfg['tds']['load_step_time']),
    pq_names=pq_names,
    pq_p_before=pq_p_before,
    pq_q_before=pq_q_before,
    pq_p_after=pq_p_after,
    pq_q_after=pq_q_after,
    M_vec=M_vec,
    D_vec=D_vec,
    success=success,
    plotter_csv=plotter_csv,
)

row['sim_id'] = 0
row['seed'] = int(cfg.get('seed', 42))
row['ed_cost'] = float(ed_cost)
row['ed_lambda'] = float(ed_lam) if ed_lam is not None else np.nan
for i, val in enumerate(Pg_opt, start=1):
    row[f'ed_Pg_{i}'] = float(val)

output_dir = Path(cfg.get('output_dir', 'experiments'))
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / cfg.get('output_csv', 'simulation_results.csv')

fieldnames = list(row.keys())
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow(row)

print(f'Wrote results to {csv_path}')


GENROU (xl <= xd2) out of typical upper limit.

   idx     | values | limit
-----------+--------+------
 GENROU_2  | 0.005  | 0.004
 GENROU_3  | 0.004  | 0.000
 GENROU_4  | 0.002  | 0.000
 GENROU_5  | 0.005  | 0.000
 GENROU_7  | 0.003  | 0.000
 GENROU_10 | 0.000  | 0.000




Wrote results to experiments/simulation_results.csv


# REAL TEST

In [145]:
import numpy as np
import cvxpy as cp
import torch
import joblib
from pathlib import Path
from models.models import MTLSharedHeads

In [146]:
model_cfg = cost_cfg["model"]
state_path = Path(model_cfg["state_dict"])
if state_path.is_dir():
    state_path = state_path / "vis_mlp_state_dict.pt"

model = MTLSharedHeads(
    in_dim=int(model_cfg["in_dim"]),
    n_tasks=int(model_cfg["n_tasks"]),
    shared_sizes=model_cfg["shared_sizes"],
    head_sizes=model_cfg["head_sizes"],
    dropout=float(model_cfg.get("dropout", 0.0)),
)
state = torch.load(state_path, map_location="cpu")
model.load_state_dict(state)
model.eval()

def _linear_layers(module):
    return [m for m in module if isinstance(m, torch.nn.Linear)]

shared_layers = [
    (m.weight.detach().cpu().numpy(), m.bias.detach().cpu().numpy())
    for m in _linear_layers(model.shared)
]
head_layers = [
    [
        (m.weight.detach().cpu().numpy(), m.bias.detach().cpu().numpy())
        for m in _linear_layers(head)
    ]
    for head in model.heads
]

Add feature bounds based on training data for neuron activation (big-M limits)

In [147]:
# x_min/x_max must be in the SAME scale used by the NN
x_min, x_max, _ = compute_feature_bounds_from_training_data(cost_cfg)

x_min = np.asarray(x_min, dtype=float)
x_max = np.asarray(x_max, dtype=float)

assert x_min.shape[0] == model_cfg["in_dim"]
assert x_max.shape[0] == model_cfg["in_dim"]

Add bounds on predicted labels (i.e. prod. of IBRs, freq. labels...)

In [148]:
def relu_big_m(z, y, a, z_min, z_max):
    # Exact ReLU with binary a
    return [
        y >= 0,
        y >= z,
        y <= z - z_min * (1 - a),
        y <= z_max * a,
    ]

x = cp.Variable(model_cfg["in_dim"], name="features")
constraints = []

# interval bounds for inputs
h_min = x_min.copy()
h_max = x_max.copy()
h = x

# shared trunk
for li, (W, b) in enumerate(shared_layers):
    z = W @ h + b
    z_min = np.maximum(W, 0) @ h_min + np.minimum(W, 0) @ h_max + b
    z_max = np.maximum(W, 0) @ h_max + np.minimum(W, 0) @ h_min + b

    y = cp.Variable(b.shape[0], name=f"shared_{li}")
    a = cp.Variable(b.shape[0], boolean=True, name=f"shared_bin_{li}")
    constraints += relu_big_m(z, y, a, z_min, z_max)

    h = y
    h_min = np.maximum(0, z_min)
    h_max = np.maximum(0, z_max)

# heads
outputs = []
for head_idx, layers in enumerate(head_layers):
    h_head = h
    hmin_head = h_min
    hmax_head = h_max

    for li, (W, b) in enumerate(layers):
        z = W @ h_head + b
        if li < len(layers) - 1:
            z_min = np.maximum(W, 0) @ hmin_head + np.minimum(W, 0) @ hmax_head + b
            z_max = np.maximum(W, 0) @ hmax_head + np.minimum(W, 0) @ hmin_head + b

            y = cp.Variable(b.shape[0], name=f"head{head_idx}_{li}")
            a = cp.Variable(b.shape[0], boolean=True, name=f"head{head_idx}_bin_{li}")
            constraints += relu_big_m(z, y, a, z_min, z_max)

            h_head = y
            hmin_head = np.maximum(0, z_min)
            hmax_head = np.maximum(0, z_max)
        else:
            y_out = cp.Variable(1, name=f"out{head_idx}")
            constraints.append(y_out == z)
            outputs.append(y_out)

y = cp.hstack(outputs)

# output bounds (scaled space)
y_min = np.asarray(cost_cfg["bounds"]["y_min"], dtype=float)
y_max = np.asarray(cost_cfg["bounds"]["y_max"], dtype=float)
constraints += [y >= y_min, y <= y_max]

/Applications/anaconda3/envs/GESTURE/lib/python3.9/site-packages/cvxpy/expressions/expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 181 times so far.

  warnings.warn(msg, UserWarning)
/Applications/anaconda3/envs/GESTURE/lib/python3.9/site-packages/cvxpy/expressions/expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 182 times so far.

  warn

In [149]:
print("NN constraints count:", len(constraints))
print("last 5 NN constraints:")
for c in constraints[-5:]:
    print(c)

NN constraints count: 82
last 5 NN constraints:
head7_1 <= [[0.02 0.01 ... -0.04 -0.00]
 [0.07 0.07 ... -0.04 0.00]
 ...
 [-0.01 -0.05 ... 0.05 0.00]
 [-0.03 -0.10 ... -0.00 0.00]] @ head7_0 + [-4.06554388e-03  8.13347250e-02  2.63056196e-02 -4.24761372e-03
 -4.50836569e-02  6.39207885e-02  3.50773856e-02  8.64704102e-02
  8.63702372e-02  1.00968387e-02 -2.31669874e-11  4.45720553e-02
 -2.22481787e-02  7.73703605e-02 -5.95008665e-09  5.54987006e-02
 -6.01652497e-03  8.83978754e-02  1.57200266e-02 -5.57263413e-24
  3.31809931e-02 -2.84744613e-13  1.54536637e-03  2.83214953e-02
  6.57084659e-02 -1.39329437e-09 -1.50009528e-01 -3.36207933e-08
  1.89448781e-02 -3.26969922e-02 -2.26889476e-02  1.28801912e-01
 -5.56392297e-02 -4.16219793e-02 -3.18992201e-13 -5.81028238e-02
  1.56445783e-02  1.05047867e-01 -2.26902887e-02  2.31388491e-02
  1.33531978e-02 -1.14935637e-03  1.00524858e-01  4.68778750e-03
  3.56173590e-02  1.32692352e-01  1.11210020e-02 -1.38804549e-03
 -2.98388764e-15 -7.8955538

Give the features values as constraints

In [150]:
base_scale = 1.0
step_scale = 1.2

case_path = cfg['case']
ss = andes.load(case_path, setup=False)
ss.config.freq = float(50)
add_measurement_devices(ss)

rng = np.random.default_rng(int(cfg.get('seed', 42)))
regcv1_ids = ss.REGCV1.name.v
M_vec = rng.uniform(cfg['ibr']['M_range'][0], cfg['ibr']['M_range'][1], size=len(regcv1_ids))
D_vec = rng.uniform(cfg['ibr']['D_range'][0], cfg['ibr']['D_range'][1], size=len(regcv1_ids))

for uid in range(ss.PQ.n):
    ss.PQ.p0.v[uid] = ss.PQ.p0.v[uid] * base_scale
    ss.PQ.q0.v[uid] = ss.PQ.q0.v[uid] * base_scale
for uid in range(ss.PV.n):
    ss.PV.p0.v[uid] = ss.PV.p0.v[uid] * base_scale
    ss.PV.q0.v[uid] = ss.PV.q0.v[uid] * base_scale

ss.REGCV1.M.v, ss.REGCV1.D.v = M_vec, D_vec

ss.PQ.config.p2p = 1
ss.PQ.config.q2q = 1
ss.PQ.config.p2z = 0
ss.PQ.config.q2z = 0
ss.PQ.config.p2i = 0
ss.PQ.config.q2i = 0
ss.PQ.config.pq2z = 0

pq_p_before = np.asarray(ss.PQ.p0.v, dtype=float).copy()
pq_q_before = np.asarray(ss.PQ.q0.v, dtype=float).copy()
pq_names = list(ss.PQ.name.v) if ss.PQ.n else []

M_agg = np.mean(np.concatenate([ss.GENROU.M.v, ss.REGCV1.M.v])).sum()
D_agg = np.mean(np.concatenate([ss.GENROU.D.v, ss.REGCV1.D.v])).sum()

features = build_feature_row(
    base_load_scale=base_scale,
    load_step_scale=step_scale,
    load_step_time=float(cfg['tds']['load_step_time']),
    pq_names=pq_names,
    pq_p_before=pq_p_before,
    pq_q_before=pq_q_before,
    pq_p_after=pq_p_before * step_scale,
    pq_q_after=pq_q_before * step_scale,
    M_vec=M_vec,
    D_vec=D_vec,
    M_agg=M_agg,
    D_agg=D_agg,
)

x_features = cost_cfg.get('features', {}).get('x_features') or list(features.keys())

In [151]:
ss = andes.load(case_path)

ng = ss.PV.n + ss.Slack.n
Pg = cp.Variable(ng)
Pd = float(np.sum(ss.PQ.p0.v))

Pg_min = ss.PV.pmin.v.tolist() + ss.Slack.pmin.v.tolist()
Pg_max = ss.PV.pmax.v.tolist() + ss.Slack.pmax.v.tolist()
Pg_now = ss.PV.p0.v.tolist() + ss.Slack.p0.v.tolist()

constraints = [
    cp.sum(Pg) == Pd,
    Pg >= Pg_min,
    Pg <= Pg_max,
]

In [152]:
# If you want to fix x to current scaled features:
feat_vec = np.array([features[name] for name in x_features], dtype=float).reshape(1, -1)
x_scaler_path = cost_cfg.get("scalers", {}).get("x_scaler_path")
if x_scaler_path:
    x_scaler = joblib.load(x_scaler_path)
    feat_scaled = x_scaler.transform(feat_vec)
else:
    feat_scaled = feat_vec

constraints += [x == feat_scaled.reshape(-1)]

/Applications/anaconda3/envs/GESTURE/lib/python3.9/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


link outputs of IBRs with real Pg val.

In [153]:
# example: y[4:8] are extra power for 4 IBRs, mapped to Pg indices
ibr_idx = [0, 5, 7, 8]   # <- indices of IBRs in Pg
P_extra = y[4:8]         # NN outputs

# extra power cannot exceed headroom
Pg_min = np.asarray(Pg_min, dtype=float)
Pg_max = np.asarray(Pg_max, dtype=float)
Pg_now = np.asarray(Pg_now, dtype=float)
constraints += [P_extra <= Pg_max[ibr_idx] - Pg_now[ibr_idx]]
constraints += [P_extra >= Pg_min[ibr_idx] - Pg_now[ibr_idx]]
constraints += [P_extra == Pg_now[ibr_idx] - Pg[ibr_idx]]


define objective (min. cost) and solve

In [154]:
a = _repeat_or_validate(cost_cfg['ed_costs']['a'], ss.PV.n + ss.Slack.n, 'ed_costs.a')
b = _repeat_or_validate(cost_cfg['ed_costs']['b'], ss.PV.n + ss.Slack.n, 'ed_costs.b')
c = _repeat_or_validate(cost_cfg['ed_costs']['c'], ss.PV.n + ss.Slack.n, 'ed_costs.c')

In [155]:
cost_expr = a + cp.multiply(b, Pg) + cp.multiply(c, cp.square(Pg))
objective = cp.Minimize(cp.sum(cost_expr))

In [156]:
prob = cp.Problem(objective, constraints)
prob.solve(solver="GUROBI")  # or "SCIP" / "GLPK_MI"
print(prob.status)
print(prob.constraints[:10])
print(prob.value)
print(Pg.value)
print(Pg.value-Pg_now)
print(y.value)

optimal
[Equality(Expression(AFFINE, UNKNOWN, ()), Constant(CONSTANT, NONNEGATIVE, ())), Inequality(Constant(CONSTANT, ZERO, (10,))), Inequality(Variable((10,), var7512)), Equality(Variable((53,), features), Constant(CONSTANT, UNKNOWN, (53,))), Inequality(Expression(AFFINE, UNKNOWN, (4,))), Inequality(Constant(CONSTANT, NONPOSITIVE, (4,))), Equality(Expression(AFFINE, UNKNOWN, (4,)), Expression(AFFINE, UNKNOWN, (4,)))]
1972.6631344929244
[8.7217277  4.76371205 4.76371205 4.76371205 4.76371205 6.97
 4.76371205 5.64       8.65       4.76371205]
[ 4.36086385 -1.69628793 -2.48628793 -1.75628793 -0.31628793  0.10000002
 -1.03628793  2.42478662  1.00216619 -0.97798795]
[ 0.          0.          0.          0.         -4.36086385 -0.10000002
 -2.42478662 -1.00216619]


In [68]:
Pd

63.77619600000001

In [53]:
ss.PQ.p0.v = ss.PQ.p0.v * 1.1

In [72]:

Pg_opt, ed_cost, ed_lam = ed_calculation(
    ss,
    a=a,
    b=b,
    c=c,
    extra_constraints=constraints,
    solver='GUROBI',
)

print('ED cost without MIP NN constraints:', ed_cost)
print('ED lambda without MIP NN constraints:', ed_lam)
print('Pg_opt without MIP NN constraints:', sum(Pg_opt))
print('Pg_opt without MIP NN constraints:', Pg_now)

ED cost without MIP NN constraints: 1776.6433884829898
ED lambda without MIP NN constraints: 97.72786666703902
Pg_opt without MIP NN constraints: 57.97836000000001
Pg_opt without MIP NN constraints: [10.4   6.5   7.5   6.82  5.5   6.97  5.95  5.64  8.65 11.  ]


In [55]:
y.value

array([0., 0., 0., 0., 0., 0., 0., 0.])

In [56]:
print("total constraints:", len(constraints))
print("last 3 constraints:")
for c in constraints[-3:]:
    print(c)

prob = cp.Problem(cp.Minimize(0), constraints)
prob.solve(solver="GUROBI")  # or SCIP/GLPK_MI
print("status:", prob.status)

if y.value is not None:
    print("y (MILP) = ", y.value)
    # print("violations:",
    #       np.any(y.value < y_min) or np.any(y.value > y_max))

total constraints: 6
last 3 constraints:
Hstack(out0, out1, out2, out3, out4, out5, out6, out7)[4:8] <= [0. 0. 0. 0.]
[-10.4   -6.97  -5.64  -8.65] <= Hstack(out0, out1, out2, out3, out4, out5, out6, out7)[4:8]
Hstack(out0, out1, out2, out3, out4, out5, out6, out7)[4:8] == [10.4   6.97  5.64  8.65] + -(var3047[0, 5, 7, 8])
status: optimal
y (MILP) =  [0. 0. 0. 0. 0. 0. 0. 0.]
